# Is a functional-evidence claim a stable category?

**A blind inter-rater check on 50 ClinVar submissions.**

A companion notebook classifies 32,569 ClinVar submissions for a RAS-MAPK panel
into three classes: the submission asserts that a functional experiment was
performed on the variant, asserts that no such experiment exists, or says
nothing on the question. The classifier was scored against 450 labelled submissions across five
sequential samples. Only some of those are evaluation: each was drawn after the
version it first tested had been frozen, but two of the five were later used to
write patterns and became training sets. The per-sample role is recorded in the
published ledger.

Those 450 labels were produced by a language model adjudicating each submission
against a written rubric. **No human curator read them.** That is a real
limitation, and the obvious way to test it is to have a human read some and see
whether the two agree.

## What this notebook found

Fifty submissions were drawn at random across the five samples with a fixed
seed, stripped of the model's verdict, and read blind by one person — the
author, who is not a trained variant curator. Then the two readings were
compared.

**They agreed on 40.4% of the 47 submissions where both committed to a class.**
Chance agreement from the observed marginals is 33.5%. Cohen's κ = 0.104.

That is not a validation. It is closer to two readers disagreeing about what the
question means.

One label in the key was corrected before scoring. A submission reading *"A
functional assay has been performed on c.775T>G (p.Ser259Ala), but … it has
not been assessed for PS3 at this time"* had been recorded as a denial, when
an assay plainly exists; the human reader called it a claim. Scoring against
the uncorrected key gives 38.3% agreement and κ = 0.076. The correction moves
the result in the classifier's favour, which is why it is stated here rather
than applied quietly — set `KEY_CORRECTIONS = {}` below to reproduce the
uncorrected figures.

## Why this is a result rather than a failure

Agreement was not uniformly low. It varied by class:

| the model said | human agreed |
|---|---:|
| claim | 8 of 16 &nbsp;(50%) |
| neither | 7 of 16 &nbsp;(44%) |
| **denial** | **4 of 15 &nbsp;(27%)** |

The unstable category is **denial**. And the reason is visible in the
submissions themselves: *"has not been previously published as pathogenic or
benign"* and *"functional studies have not been performed"* are different
statements, and a reader can take either as denying functional evidence.

The first is about the clinical literature. The second is about experiments.
Under the ACMG framework they belong to different criteria entirely — PS4 and
PS3 — and a classifier, a curator, or a panel that conflates them will reach
different conclusions from the same text.

## What this does and does not undermine

It does not touch the findings that come from the structured record rather than
from these labels: that 32.4% of submissions carry no comment at all, that 147
assert functional evidence through the code `PS3` alone with no prose sentence
doing so, and that the ACMG framework provides a code for asserting functional
evidence while providing none for having looked and found nothing.

It does bear on any figure computed from the three classes — the claim and
denial rates, the per-laboratory table, the panel ratio. Those should be read
with this κ beside them.

## Provenance

Every label, every verdict and the free-text reasoning behind each one is
published, so a reader can disagree line by line rather than take the
classification on trust. The five samples, the rubric and this blind read are in
the accompanying dataset.

**An invitation, not a conclusion.** If the distinction between denying
functional evidence and merely lacking it is unstable between two readers, it is
worth knowing whether it is stable among curators who do this professionally.
The same fifty submissions are below, and the comparison cell will score anyone
else's reading against both.

## Method

Fifty submissions drawn with `random_state=20260915` from the pooled 450
labelled rows, deduplicated by SCV. The model's verdict was written to a
separate file and not opened until the reading was complete.

The rubric, frozen before reading:

- **claim** — the submission says an experiment was performed, bearing on this
  variant
- **denial** — it says such evidence is absent, lacking, or unconfirmed
- **neither** — a predictor score, a vendor model, a structural inference, a
  clinical observation, or silence on the question

The adjudicator's thirteen verdict labels collapse onto the same three classes.
Rows where either side declined to commit are counted separately rather than
scored as disagreements, since they are disagreements about whether the text
decides at all.

In [1]:
import pandas as pd, numpy as np
from pathlib import Path

BASE = Path("/kaggle/input/datasets/fernandosr85/clinvar-claim-audit-labels")
FILES = ["panel_claims_audit_sample130_rev.csv",
         "panel_claims_v2_heldout_110_rev.csv",
         "panel_claims_v3_heldout_90_rev.csv",
         "panel_claims_v4_negatives_60.csv",
         "panel_claims_v5_negatives_60.csv"]

frames = []
for f in FILES:
    p = BASE / f
    if not p.exists():
        print(f"not found: {f}")
        continue
    d = pd.read_csv(p)
    col = next((c for c in ("text", "Description") if c in d.columns), None)
    frames.append(pd.DataFrame({
        "source": f.replace("panel_claims_", "").replace("_rev.csv", ""),
        "scv": d.get("SCV", pd.Series(range(len(d)))).astype(str),
        "submitter": d.get("Submitter", ""),
        "text": d[col].fillna("").astype(str),
        "model_verdict": d["audit_verdict"],
    }))

pool = pd.concat(frames, ignore_index=True).drop_duplicates("scv")
print(f"{len(pool)} labelled rows, from {len(frames)} samples")
print(pool["source"].value_counts().to_string())

# Fixed seed, so the same fifty rows come back on a re-run and the selection
# cannot be quietly redrawn until it agrees.
sample = pool.sample(50, random_state=20260915).reset_index(drop=True)

# The verdict goes to a separate file. Seeing it before deciding would make the
# exercise measure agreement with what you already read, not independent
# judgement - which is the whole point of doing this.
sample[["source", "scv", "model_verdict"]].to_csv("blind_key.csv", index=False)
sample[["scv", "submitter", "text"]].assign(my_verdict="").to_csv(
    "blind_read.csv", index=False)

print()
print("Read the fifty submissions below, then type one character each into")
print("MY_READ further down. That string is the reading of record: it lives in")
print("the notebook, so anyone can re-check it. blind_read.csv is written as a")
print("convenience for reading away from the notebook, but nothing reads it back.")
print()
print("  c  claim   - says an experiment was performed, bearing on this variant")
print("  d  denial  - says such evidence is absent, lacking, or unconfirmed")
print("  n  neither - a predictor score, a vendor model, a structural inference,")
print("               a clinical observation, or silence on the question")
print("  ?  unsure  - the text does not decide it")
print()
print("Do not open blind_key.csv until the reading is finished.")

450 labelled rows, from 5 samples
source
audit_sample130        130
v2_heldout_110         110
v3_heldout_90           90
v4_negatives_60.csv     60
v5_negatives_60.csv     60

Read the fifty submissions below, then type one character each into
MY_READ further down. That string is the reading of record: it lives in
the notebook, so anyone can re-check it. blind_read.csv is written as a
convenience for reading away from the notebook, but nothing reads it back.

  c  claim   - says an experiment was performed, bearing on this variant
  d  denial  - says such evidence is absent, lacking, or unconfirmed
  n  neither - a predictor score, a vendor model, a structural inference,
               a clinical observation, or silence on the question
  ?  unsure  - the text does not decide it

Do not open blind_key.csv until the reading is finished.


In [2]:
from IPython.display import HTML, display
import html as _html, re as _re

# Terms worth spotting quickly. Highlighting is a reading aid and nothing more:
# a highlighted word does not decide the verdict, and the hardest rows in this
# sample are hard precisely because the vocabulary points the wrong way.
_CUE = _re.compile(
    r"\b(functional|experimental|in.?vitro|in.?vivo|assay|assays|"
    r"PS3|BS3|zebrafish|phosphatase|kinase activity|RNA studies|"
    r"in silico|REVEL|CADD|PolyPhen|SIFT|AlphaMissense|SpliceAI|"
    r"Evidence Modeling|not been|no published|unknown|lacking|"
    r"another|wild.?type|same (codon|residue|position))\b", _re.I)

_WARM = {"functional", "experimental", "in vitro", "in-vitro", "in vivo",
         "in-vivo", "assay", "assays", "ps3", "bs3", "zebrafish",
         "phosphatase", "rna studies", "kinase activity"}
_COOL = {"in silico", "revel", "cadd", "polyphen", "sift", "alphamissense",
         "spliceai", "evidence modeling"}


def _mark(text):
    def sub(m):
        w = m.group(0); k = w.lower()
        c = "#c0392b" if k in _WARM else "#2471a3" if k in _COOL else "#7d6608"
        return f"<b style='color:{c}'>{_html.escape(w)}</b>"
    return _CUE.sub(sub, _html.escape(str(text)))


_CSS = """<style>
.br{font-family:-apple-system,Segoe UI,Roboto,sans-serif;max-width:1000px}
.br-h{background:#1b2631;color:#fff;padding:15px 20px;border-radius:6px 6px 0 0}
.br-h h3{margin:0;font-size:15px;font-weight:600;letter-spacing:.2px}
.br-h p{margin:8px 0 0;font-size:12px;opacity:.85;line-height:1.6}
.br-lg{display:flex;gap:20px;margin-top:11px;font-size:11px;flex-wrap:wrap;
       padding-top:9px;border-top:1px solid rgba(255,255,255,.15)}
.br-c{border:1px solid #dfe4ea;border-top:none;padding:15px 20px}
.br-c:nth-child(even){background:#fafbfc}
.br-n{display:inline-block;background:#1b2631;color:#fff;font-weight:700;
      border-radius:4px;padding:2px 10px;font-size:12px;margin-right:11px}
.br-l{color:#566573;font-size:11px;text-transform:uppercase;letter-spacing:.5px}
.br-t{margin-top:10px;font-size:13px;line-height:1.7;color:#1c2833}
.br-f{background:#f4f6f7;border:1px solid #dfe4ea;border-top:none;
      padding:13px 20px;font-size:12px;color:#566573;line-height:1.6;
      border-radius:0 0 6px 6px}
</style>"""


def show_blind(sample, start=1, end=25):
    out = [_CSS, "<div class='br'>",
           "<div class='br-h'><h3>Blind read &mdash; did somebody run an "
           "experiment on THIS variant?</h3>"
           "<p><b>claim</b> &nbsp;an experiment was done, bearing on this "
           "variant<br>"
           "<b>denial</b> &nbsp;such evidence is absent, lacking, or "
           "unconfirmed<br>"
           "<b>neither</b> &nbsp;a predictor score, a vendor model, a "
           "structural inference, or silence on the question</p>"
           "<div class='br-lg'>"
           "<span style='color:#e74c3c'><b>red</b> &nbsp;experiment words</span>"
           "<span style='color:#5dade2'><b>blue</b> &nbsp;prediction words</span>"
           "<span style='color:#d4ac0d'><b>amber</b> &nbsp;absence or "
           "attribution</span></div></div>"]
    for i, r in sample.iloc[start-1:end].iterrows():
        out.append(f"<div class='br-c'><span class='br-n'>{i+1}</span>"
                   f"<span class='br-l'>{_html.escape(str(r['submitter'])[:60])}"
                   f"</span><div class='br-t'>{_mark(r['text'])}</div></div>")
    out.append("<div class='br-f'>Highlighting is a reading aid, not a verdict. "
               "Rows carrying experiment vocabulary are often still "
               "<b>neither</b>: a vendor model citing in-vitro data, an and/or "
               "list that never says which was done, or an assay performed on a "
               "neighbouring variant.</div></div>")
    display(HTML("".join(out)))


show_blind(sample, 1, 25)

In [3]:
show_blind(sample, 26, 50)

In [4]:
# One character per submission, in order, no separators needed:
#   c = claim      d = denial      n = neither      ? = genuinely unsure
#
# Typed rather than clicked on purpose: the string lives in the notebook, so the
# reading is part of the record and can be re-checked by anyone. A widget's
# state would not survive a save.
# The author's reading, frozen. A second reader replaces MY_READ below;
# this string stays put so the two can still be compared to each other.
AUTHOR_READ = """
cndcn ndccd nnncd dcnnc ccdnn
ndccd cnnnd dccnn cnndc ddncc
"""

import pandas as pd

_MAP = {"c": "claim", "d": "denial", "n": "neither", "?": "unsure"}
# Replace this with your own fifty characters to be scored against both the
# adjudicator and the author. Leave it as-is to reproduce the published run.
MY_READ = AUTHOR_READ

mine = [_MAP[ch] for ch in MY_READ.replace(" ", "").replace("\n", "") if ch in _MAP]
author = [_MAP[ch] for ch in AUTHOR_READ.replace(" ", "")
          .replace(chr(10), "") if ch in _MAP]

if len(mine) != len(sample):
    raise SystemExit(f"{len(mine)} verdicts for {len(sample)} rows - "
                     "fill every position before running this")

key = pd.read_csv("blind_key.csv")

# Collapse the adjudicator's thirteen labels onto the same three classes.
_COLLAPSE = {
    # adjudicated as a claim
    "TRUE_CLAIM": "claim", "MISSED_CLAIM": "claim",
    "TRUE_BOTH_claim_ok": "claim", "TRUE_CLAIM_BORDERLINE": "claim",
    # adjudicated as a denial. FALSE_CLAIM means the pattern said claim and the
    # adjudicator overruled it to denial - the adjudicated class is what counts.
    "TRUE_DENIAL": "denial", "MISSED_DENIAL": "denial", "FALSE_CLAIM": "denial",
    # adjudicated as neither. BY_ACCIDENT means the classifier reached the right
    # class through a wrong route; the class is still neither.
    "CORRECT_NEITHER": "neither", "CORRECT_UNLABELLED": "neither",
    "FALSE_DENIAL": "neither", "CORRECT_NEITHER_BY_ACCIDENT": "neither",
    # the adjudicator declined to decide
    "AMBIGUOUS": "unsure", "FALSE_CLAIM_DEBATABLE": "unsure",
}

_unmapped = sorted(set(key["model_verdict"]) - set(_COLLAPSE))
if _unmapped:
    print(f"note: {_unmapped} not in the collapse map, counted as unsure\n")

key["model"] = key["model_verdict"].map(_COLLAPSE).fillna("unsure")

# One label in the key is known to be wrong, and it was found by the audit trail
# rather than by this reading. SCV001424749.1 reads "A functional assay has been
# performed on c.775T>G (p.Ser259Ala), but ... it has not been assessed for PS3
# at this time". An assay exists, so the class is claim; the adjudicator recorded
# FALSE_CLAIM, which collapses to denial.
#
# Corrected here rather than quietly, because it moves the result in the
# classifier's favour - 38.3% to 40.4% agreement, kappa 0.076 to 0.104 - and a
# correction that helps you is the one to be loudest about. Set to {} to
# reproduce the uncorrected figures.
KEY_CORRECTIONS = {"SCV001424749.1": "claim"}

_fixed = key["scv"].astype(str).isin(KEY_CORRECTIONS)
if _fixed.any():
    key.loc[_fixed, "model"] = (key.loc[_fixed, "scv"].astype(str)
                                .map(KEY_CORRECTIONS))
    print(f"{int(_fixed.sum())} key label(s) corrected before scoring: "
          f"{sorted(KEY_CORRECTIONS)}")
    print()

cmp = sample[["scv", "submitter", "text"]].copy()
cmp["mine"] = mine
cmp["author"] = author
cmp["model"] = key["model"].values
cmp["raw"] = key["model_verdict"].values
cmp["agree"] = cmp["mine"] == cmp["model"]

# Rows where either side said "unsure" are not disagreements about the text;
# they are disagreements about whether the text decides. Counted separately.
decided = cmp[(cmp["mine"] != "unsure") & (cmp["model"] != "unsure")]
n_agree = int(decided["agree"].sum())

print("=" * 66)
print(f"{len(cmp)} submissions read blind")
print(f"  both sides committed to a class : {len(decided)}")
print(f"  agreed                          : {n_agree} "
      f"({100*n_agree/max(1,len(decided)):.0f}%)")
print(f"  one side unsure                 : {len(cmp) - len(decided)}")
print("=" * 66)

print("\nby class, as you read them:")
print(pd.crosstab(cmp["mine"], cmp["model"],
                  rownames=["you"], colnames=["model"]).to_string())

# Only the decided rows are disagreements. A row where the adjudicator declined
# to commit is a disagreement about whether the text decides at all, and it was
# already counted separately above; listing it here as well would charge the
# same row twice against the same reading.
dis = decided[~decided["agree"]]
unsure_rows = cmp[(cmp["mine"] == "unsure") | (cmp["model"] == "unsure")]
if len(dis):
    print()
    print(f"{len(dis)} disagreement(s) on the decided rows - read these again,")
    print(f"plus {len(unsure_rows)} row(s) where one side declined to commit:")
    print()
    for _, r in dis.iterrows():
        # The adjudicator's raw label is deliberately not printed here. It was
        # what the reading was testing, and showing it beside the text steers
        # anyone who re-reads the disagreements later. It stays in the saved CSV.
        print(f"  you={r['mine']:8s} model={r['model']:8s}")
        print(f"    {' '.join(str(r['text']).split())[:190]}")
        print()

cmp.to_csv("blind_comparison.csv", index=False)
print("saved: blind_comparison.csv")
print(f"""
This is the figure to report: agreement between one untrained human reading and
the LLM adjudication that produced all 450 labels, on {len(decided)} submissions
drawn at random across the five samples with a fixed seed.

It does not make the labels human-curated. It measures how far one human agrees
with them, and publishes the disagreements so a reader can judge for themselves.""")

1 key label(s) corrected before scoring: ['SCV001424749.1']

50 submissions read blind
  both sides committed to a class : 47
  agreed                          : 19 (40%)
  one side unsure                 : 3

by class, as you read them:
model    claim  denial  neither  unsure
you                                    
claim        8       5        5       0
denial       4       4        4       1
neither      4       6        7       2

28 disagreement(s) on the decided rows - read these again,
plus 3 row(s) where one side declined to commit:

  you=neither  model=claim   
    This sequence change replaces glycine, which is neutral and non-polar, with valine, which is neutral and non-polar, at codon 60 of the HRAS protein (p.Gly60Val). This variant is not present 

  you=claim    model=neither 
    In summary, the available evidence is currently insufficient to determine the role of this variant in disease. Therefore, it has been classified as a Variant of Uncertain Significance. Algor



In [5]:
# Inter-rater statistics. Raw agreement alone is not interpretable: with three
# classes it looks like a percentage but its floor depends on how often each
# reader used each class. Cohen's kappa corrects for that, and the confidence
# interval says how little 47 observations settle.
import numpy as np
import pandas as pd
from math import sqrt

CLASSES = ["claim", "denial", "neither"]


def kappa_report(left, right, title, right_name="the adjudicator"):
    """Agreement between two readings, over the rows where both committed.

    The matrix is derived from the readings, never transcribed. An earlier
    version typed it out by hand: it happened to be right, but any edit to a
    reading would have moved the data while leaving those nine numbers alone,
    and the published kappa would have described a reading nobody performed.
    """
    pair = pd.DataFrame({"left": list(left), "right": list(right)})
    pair = pair[(pair["left"] != "unsure") & (pair["right"] != "unsure")]
    ct = pd.crosstab(pair["left"], pair["right"])
    M = np.array([[int(ct.loc[r, c]) if (r in ct.index and c in ct.columns)
                   else 0 for c in CLASSES] for r in CLASSES])
    n = int(M.sum())
    if not n:
        print(f"{title}: nothing to compare")
        return None

    po = float(np.trace(M) / n)
    pe = float(sum(M.sum(axis=1)[i] * M.sum(axis=0)[i] for i in range(3)) / n**2)
    kappa = (po - pe) / (1 - pe)

    # Wilson interval - the normal approximation is unreliable at this n.
    z = 1.96
    den = 1 + z*z/n
    centre = (po + z*z/(2*n)) / den
    half = z * sqrt(po*(1-po)/n + z*z/(4*n*n)) / den

    print(title)
    print(pd.DataFrame(M, index=CLASSES, columns=CLASSES).to_string())
    print()
    print(f"  n = {n} submissions where both readers committed to a class")
    print(f"  observed agreement : {100*po:.1f}%  "
          f"(95% CI {100*(centre-half):.1f}% to {100*(centre+half):.1f}%)")
    print(f"  chance agreement   : {100*pe:.1f}%  (from the observed marginals)")
    print(f"  Cohen's kappa      : {kappa:.3f}")
    print()
    print(f"  agreement by the class {right_name} assigned:")
    for j, name in enumerate(CLASSES):
        col = M[:, j]
        if col.sum():
            print(f"    {name:8s} {col[j]:2d} of {col.sum():2d}  "
                  f"({100*col[j]/col.sum():.0f}%)")
    print()
    return M


M = kappa_report(cmp["mine"], cmp["model"],
                 "This reading against the adjudicator that produced all 450 "
                 "labels:")

# The invitation at the top promises to score a new reader against both the
# adjudicator and the author. That only works if the author's reading survives
# being replaced, which is why it is kept as AUTHOR_READ rather than overwritten.
if MY_READ.strip() != AUTHOR_READ.strip():
    kappa_report(cmp["mine"], cmp["author"],
                 "This reading against the author's - two humans on the same "
                 "fifty rows, which is the comparison a curator would want:",
                 right_name="the author")
else:
    print("MY_READ is still the author's reading, so there is no second human to")
    print("compare against. Replace MY_READ with your own fifty characters and")
    print("this cell prints a human-against-human kappa as well - which is the")
    print("number the invitation at the top is actually asking for.")
    print()

worst = CLASSES[int(np.argmin([M[j, j] / max(1, M[:, j].sum())
                               for j in range(3)]))]
print(f"The unstable class is {worst}. That is the finding: not that one reader")
print("was careless, but that the boundary between saying functional evidence is")
print("absent and simply not mentioning it does not survive two independent")
print("readings.")
print()
print("Do not report this as a validation of the labels. Report it as what it")
print("is - a measurement of how far one untrained reader agrees with the")
print("adjudication, with the disagreements published so anyone can judge for")
print("themselves.")

This reading against the adjudicator that produced all 450 labels:
         claim  denial  neither
claim        8       5        5
denial       4       4        4
neither      4       6        7

  n = 47 submissions where both readers committed to a class
  observed agreement : 40.4%  (95% CI 27.6% to 54.7%)
  chance agreement   : 33.5%  (from the observed marginals)
  Cohen's kappa      : 0.104

  agreement by the class the adjudicator assigned:
    claim     8 of 16  (50%)
    denial    4 of 15  (27%)
    neither   7 of 16  (44%)

MY_READ is still the author's reading, so there is no second human to
compare against. Replace MY_READ with your own fifty characters and
this cell prints a human-against-human kappa as well - which is the
number the invitation at the top is actually asking for.

The unstable class is denial. That is the finding: not that one reader
was careless, but that the boundary between saying functional evidence is
absent and simply not mentioning it does not surviv